# Fit productivity models

PDP `fit` task consuming `prepare/output/df_traj_all.csv`.

This notebook fits or registers:

1. ARW4
2. Unfitted GRW
3. GRW-Y
4. AR(1)-GRW-Y
5. AR(1)-GRW-S
6. Hurdle-AR(1)-GRW-S
7. Hurdle-AR(1)-GRW-S-P
8. Hurdle-AR(3)-GRW-S-P

Each model writes the parameter tables required by simulation to `fit/output/<model_tag>/`.


In [1]:
# Cell 1: imports, paths, and constants

import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from scipy.linalg import solve_discrete_lyapunov
from scipy.optimize import minimize, minimize_scalar
TASK_ROOT = Path.cwd().parent
INPUT = TASK_ROOT / 'input'
OUTPUT = TASK_ROOT / 'output'
OUTPUT.mkdir(parents=True, exist_ok=True)
EPS = 0.49
Y = 20
MIN_PROB = 1e-12
CANONICAL_STAGE_ORDER = ['0', '1-4', '5-7', '8-20']
CANONICAL_STAGE_BOUNDS = {'0': (0, 0), '1-4': (1, 4), '5-7': (5, 7), '8-20': (8, 19)}
ARW4_STAGES = [('0-4', 0, 4), ('5-7', 5, 7), ('8-13', 8, 13), ('14-19', 14, 19)]


In [2]:
# Cell 2: load prepared trajectories and construct fit frames

df_traj_all = pd.read_csv(INPUT / 'df_traj_all.csv')
df_traj_all = df_traj_all.drop(columns=[c for c in df_traj_all.columns if c.startswith('Unnamed:')], errors='ignore')
required_columns = {'dblp_id', 'CareerAge', 'CareerAgeZero', 'pubs_adj'}
missing_columns = required_columns.difference(df_traj_all.columns)
if missing_columns:
    raise ValueError(f'df_traj_all.csv is missing: {sorted(missing_columns)}')
working_df = df_traj_all.copy()
working_df['CareerAgeRaw'] = working_df['CareerAge']
working_df['CareerAge'] = working_df['CareerAgeZero'].astype(int)
working_df = working_df.sort_values(['dblp_id', 'CareerAge']).copy()
grouped = working_df.groupby('dblp_id', sort=False)
working_df['log_pubs_adj'] = np.log(working_df['pubs_adj'] + EPS)
working_df['pubs_adj_next_model'] = grouped['pubs_adj'].shift(-1)
working_df['CareerAge_next_model'] = grouped['CareerAge'].shift(-1)
working_df['log_pubs_next'] = grouped['log_pubs_adj'].shift(-1)
working_df['log_delta'] = working_df['log_pubs_next'] - working_df['log_pubs_adj']
working_df['state'] = working_df['pubs_adj'].gt(0).astype(int)
working_df['next_state'] = working_df['pubs_adj_next_model'].gt(0).astype(int)
working_df['pubs_adj_lag_1'] = grouped['pubs_adj'].shift(1)
working_df['pubs_adj_lag_2'] = grouped['pubs_adj'].shift(2)
working_df['CareerAge_lag_1'] = grouped['CareerAge'].shift(1)
working_df['CareerAge_lag_2'] = grouped['CareerAge'].shift(2)
working_df['state_lag_1'] = grouped['state'].shift(1)
working_df['state_lag_2'] = grouped['state'].shift(2)
is_next_year = working_df['CareerAge_next_model'].eq(working_df['CareerAge'] + 1)
is_in_window = working_df['CareerAge'].between(0, Y - 1)
working_df_fit = working_df.loc[is_next_year & is_in_window].dropna(subset=['pubs_adj', 'pubs_adj_next_model', 'log_pubs_adj', 'log_pubs_next']).copy()
working_df_fit['fit_stage'] = pd.cut(working_df_fit['CareerAge'], bins=[-1, 0, 4, 7, 19], labels=CANONICAL_STAGE_ORDER).astype(str)
working_df_fit_ar2 = working_df_fit.loc[working_df_fit['CareerAge_lag_1'].eq(working_df_fit['CareerAge'] - 1)].copy()
working_df_fit_ar3 = working_df_fit.loc[working_df_fit['CareerAge_lag_1'].eq(working_df_fit['CareerAge'] - 1) & working_df_fit['CareerAge_lag_2'].eq(working_df_fit['CareerAge'] - 2)].copy()
q0_empirical = working_df.loc[working_df['CareerAge'].eq(0), 'pubs_adj'].dropna()
_, alpha_q0 = stats.expon.fit(q0_empirical, floc=0)
shared_initialization_params = pd.DataFrame([{'distribution': 'Exponential(loc=0)', 'n': len(q0_empirical), 'scale': alpha_q0}])
print(f'Prepared rows: {len(working_df):,}')
print(f'Valid transitions: {len(working_df_fit):,}')
print(f"Scholars: {working_df['dblp_id'].nunique():,}")


Prepared rows: 29,119
Valid transitions: 27,033
Scholars: 2,086


In [3]:
# Cell 3: shared fit and export helpers

def canonical_stage(year):
    if year == 0:
        return '0'
    if year <= 4:
        return '1-4'
    if year <= 7:
        return '5-7'
    return '8-20'

def save_tables(model_tag, tables):
    model_output = OUTPUT / model_tag
    model_output.mkdir(parents=True, exist_ok=True)
    for filename, table in tables.items():
        table.to_csv(model_output / filename, index=False)

def fit_normal(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {'n': 0, 'mu': np.nan, 'sigma': np.nan, 'var': np.nan}
    mu, sigma = stats.norm.fit(values)
    return {'n': len(values), 'mu': mu, 'sigma': sigma, 'var': sigma ** 2}

def fit_log_ar1(subset):
    x = subset['log_pubs_adj'].to_numpy(dtype=float)
    y = subset['log_pubs_next'].to_numpy(dtype=float)
    if len(y) < 2:
        return {'n': len(y), 'intercept': np.nan, 'beta': np.nan, 'sigma_resid': np.nan, 'var_resid': np.nan, 'implied_stationary_mean_log': np.nan, 'implied_stationary_var_log': np.nan, 'mean_resid': np.nan, 'sd_resid': np.nan}
    X = np.column_stack([np.ones(len(y)), x])
    intercept, beta = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - intercept - beta * x
    var = np.mean(resid ** 2)
    sigma = np.sqrt(var)
    if abs(beta) < 1:
        implied_mean = intercept / (1 - beta)
        implied_var = var / (1 - beta ** 2)
    else:
        implied_mean = np.nan
        implied_var = np.nan
    return {'n': len(y), 'intercept': intercept, 'beta': beta, 'sigma_resid': sigma, 'var_resid': var, 'implied_stationary_mean_log': implied_mean, 'implied_stationary_var_log': implied_var, 'mean_resid': resid.mean(), 'sd_resid': resid.std(ddof=0)}

def ar_stationary_moments(intercept, betas, innovation_var):
    betas = np.asarray(betas, dtype=float)
    p = len(betas)
    companion = np.zeros((p, p), dtype=float)
    companion[0] = betas
    if p > 1:
        companion[1:, :-1] = np.eye(p - 1)
    spectral_radius = float(np.max(np.abs(np.linalg.eigvals(companion))))
    is_stationary = spectral_radius < 1
    if not is_stationary:
        return (spectral_radius, False, np.nan, np.nan)
    implied_mean = intercept / (1 - betas.sum())
    innovation = np.zeros((p, p), dtype=float)
    innovation[0, 0] = innovation_var
    covariance = solve_discrete_lyapunov(companion, innovation)
    implied_var = float(covariance[0, 0])
    return (spectral_radius, True, implied_mean, implied_var)

def fit_positive_arp(subset, order):
    lag_columns = ['pubs_adj'] + [f'pubs_adj_lag_{lag}' for lag in range(1, order)]
    x = np.column_stack([np.log(subset[column].to_numpy(dtype=float)) for column in lag_columns])
    y = np.log(subset['pubs_adj_next_model'].to_numpy(dtype=float))
    keys = ['intercept'] + [f'beta_{i}' for i in range(1, order + 1)]
    empty = {'n': len(y), **{key: np.nan for key in keys}, 'sigma_resid': np.nan, 'var_resid': np.nan, 'spectral_radius': np.nan, 'is_stationary': False, 'implied_stationary_mean_log': np.nan, 'implied_stationary_var_log': np.nan}
    if len(y) < order + 1:
        return empty
    X = np.column_stack([np.ones(len(y)), x])
    coefficients = np.linalg.lstsq(X, y, rcond=None)[0]
    intercept = coefficients[0]
    betas = coefficients[1:]
    resid = y - X @ coefficients
    var = np.mean(resid ** 2)
    sigma = np.sqrt(var)
    spectral_radius, is_stationary, implied_mean, implied_var = ar_stationary_moments(intercept, betas, var)
    return {'n': len(y), 'intercept': intercept, **{f'beta_{i}': betas[i - 1] for i in range(1, order + 1)}, 'sigma_resid': sigma, 'var_resid': var, 'spectral_radius': spectral_radius, 'is_stationary': is_stationary, 'implied_stationary_mean_log': implied_mean, 'implied_stationary_var_log': implied_var}

def transition_probabilities(subset, fallback):
    counts = pd.crosstab(subset['state'], subset['next_state'])
    counts = counts.reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    fallback_counts = pd.crosstab(fallback['state'], fallback['next_state'])
    fallback_counts = fallback_counts.reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    out = {}
    for state in [0, 1]:
        row = counts.loc[state]
        if row.sum() == 0:
            row = fallback_counts.loc[state]
        probabilities = row / row.sum()
        out[f'n_from_{state}'] = int(row.sum())
        out[f'p_{state}0'] = probabilities.loc[0]
        out[f'p_{state}1'] = probabilities.loc[1]
    return out

def fit_dropout_parameters(frame):
    active_dropout = frame.loc[frame['state'].eq(1)].copy()
    active_dropout['drops_to_zero'] = active_dropout['next_state'].eq(0).astype(int)
    q_scale = float(active_dropout['pubs_adj'].median())
    active_dropout['q_scaled'] = active_dropout['pubs_adj'] / q_scale
    stage_index = {stage: i for i, stage in enumerate(CANONICAL_STAGE_ORDER)}
    active_dropout['stage_index'] = active_dropout['fit_stage'].map(stage_index).astype(int)
    initial_intercepts = []
    for stage in CANONICAL_STAGE_ORDER:
        outcomes = active_dropout.loc[active_dropout['fit_stage'].eq(stage), 'drops_to_zero']
        p = np.clip(outcomes.mean(), 1e-05, 1 - 1e-05)
        initial_intercepts.append(np.log(p / (1 - p)))

    def dropout_nll(theta):
        intercepts = theta[:-1]
        gamma = np.exp(theta[-1])
        eta = intercepts[active_dropout['stage_index'].to_numpy()] - gamma * active_dropout['q_scaled'].to_numpy()
        eta = np.clip(eta, -40, 40)
        p = 1 / (1 + np.exp(-eta))
        y = active_dropout['drops_to_zero'].to_numpy(dtype=float)
        return -np.sum(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))
    result = minimize(dropout_nll, x0=np.r_[initial_intercepts, np.log(0.1)], method='BFGS')
    intercepts = result.x[:-1]
    gamma = float(np.exp(result.x[-1]))
    rows = []
    for i, stage in enumerate(CANONICAL_STAGE_ORDER):
        p_at_scale = 1 / (1 + np.exp(-(intercepts[i] - gamma)))
        rows.append({'stage': stage, 'dropout_intercept': intercepts[i], 'dropout_gamma_positive': gamma, 'productivity_scale': q_scale, 'p_dropout_at_scale': p_at_scale})
    optimizer = pd.DataFrame([{'success': result.success, 'objective': result.fun, 'message': str(result.message)}])
    return (pd.DataFrame(rows), optimizer)

def hurdle_initialization(frame, transitions):
    q0 = frame.loc[frame['CareerAge'].eq(0), 'pubs_adj'].dropna()
    p_init_active = q0.gt(0).mean()
    alpha_q0_pos = q0.loc[q0.gt(0)].mean()
    restart_values = transitions.loc[transitions['state'].eq(0) & transitions['next_state'].eq(1), 'pubs_adj_next_model']
    alpha_restart = restart_values.mean()
    if not np.isfinite(alpha_restart) or alpha_restart <= 0:
        alpha_restart = alpha_q0_pos
    return pd.DataFrame([{'p_init_active': p_init_active, 'positive_q0_exponential_scale': alpha_q0_pos, 'restart_exponential_scale': alpha_restart, 'n_q0': len(q0), 'n_restarts': len(restart_values)}])

def arw4_stage(year):
    for label, start, end in ARW4_STAGES:
        if start <= year <= end:
            return label
    raise KeyError(year)

def trunclaplace_negloglike(alpha, values, modes):
    if not np.isfinite(alpha) or alpha <= 0:
        return np.inf
    values = np.asarray(values, dtype=float)
    modes = np.asarray(modes, dtype=float)
    log_density = -np.log(2 * alpha) - np.abs(values - modes) / alpha
    survival = 1 - stats.laplace.cdf(0, loc=modes, scale=alpha)
    survival = np.clip(survival, MIN_PROB, 1)
    return -np.sum(log_density - np.log(survival))

def fit_trunc_laplace(values, modes):
    values = np.asarray(values, dtype=float)
    modes = np.asarray(modes, dtype=float)
    scale_hi = max(10.0, 5 * np.std(values), 5 * np.mean(values), 500.0)
    result = minimize_scalar(trunclaplace_negloglike, args=(values, modes), bounds=(1e-08, scale_hi), method='bounded')
    return float(result.x)

def fit_arw4_block(subset, prior_beta=None):
    x = subset['pubs_adj'].to_numpy(dtype=float)
    y = subset['pubs_adj_next_model'].to_numpy(dtype=float)
    if prior_beta is None:
        denominator = max(np.median(x), EPS)
        prior_beta = float(np.clip(np.median(y) / denominator, 1e-06, 5.0))
    alpha0 = fit_trunc_laplace(y, prior_beta * x)
    result = minimize_scalar(lambda beta: trunclaplace_negloglike(alpha0, y, beta * x), bounds=(1e-06, 5.0), method='bounded')
    beta = float(result.x)
    alpha = fit_trunc_laplace(y, beta * x)
    return {'n': len(x), 'alpha': alpha, 'mode_beta': beta, 'nll': trunclaplace_negloglike(alpha, y, beta * x), 'mean_q_now': x.mean(), 'mean_q_next': y.mean(), 'mean_raw_delta': np.mean(y - x), 'median_raw_delta': np.median(y - x)}


In [4]:
# Cell 4: full fit, ARW4

global_arw4_fit = fit_arw4_block(working_df_fit)
arw4_fit = working_df_fit.copy()
arw4_fit['arw4_stage'] = arw4_fit['CareerAge'].map(arw4_stage)
arw4_rows = []
for label, start, end in ARW4_STAGES:
    subset = arw4_fit.loc[arw4_fit['arw4_stage'].eq(label)]
    arw4_rows.append({'stage': label, 'transition_year_start': start, 'transition_year_end': end, **fit_arw4_block(subset, prior_beta=global_arw4_fit['mode_beta'])})
arw4_stage_params = pd.DataFrame(arw4_rows)
arw4_global_params = pd.DataFrame([{'stage': 'global', **global_arw4_fit}])
arw4_initialization_params = shared_initialization_params.copy()
save_tables('arw4', {'stage-parameters.csv': arw4_stage_params, 'global-parameters.csv': arw4_global_params, 'initialization-parameters.csv': arw4_initialization_params})
print('ARW4')
display(arw4_stage_params)


ARW4


,stage,transition_year_start,transition_year_end,n,alpha,mode_beta,nll,mean_q_now,mean_q_next,mean_raw_delta,median_raw_delta
0,0-4,0,4,9789,4.493032,0.794375,27068.423528,6.634466,7.235996,0.601531,0.000000
1,5-7,5,7,5140,4.116321,0.748148,13912.655107,7.388392,7.060129,-0.328263,-0.171862
2,8-13,8,13,7664,3.797657,0.748148,20025.757747,6.642415,6.473341,-0.169074,-0.092021
3,14-19,14,19,4440,3.608783,0.774203,11397.449624,6.533666,6.423050,-0.110616,-0.060075


In [5]:
# Cell 5: full registration, unfitted GRW

unfitted_grw_params = pd.DataFrame([{'parameter': 'x0', 'value': 4.0}, {'parameter': 'log_gamma_mean', 'value': 0.0}, {'parameter': 'log_gamma_variance', 'value': 0.25}, {'parameter': 'log_gamma_sigma', 'value': 0.5}])
save_tables('unfitted_grw', {'fixed-parameters.csv': unfitted_grw_params})
print('Unfitted GRW')
display(unfitted_grw_params)


Unfitted GRW


,parameter,value
0,x0,4.00
1,log_gamma_mean,0.00
2,log_gamma_variance,0.25
3,log_gamma_sigma,0.50


In [6]:
# Cell 6: full fit, GRW-Y

grw_y_rows = []
for year in range(Y):
    subset = working_df_fit.loc[working_df_fit['CareerAge'].eq(year), 'log_delta']
    grw_y_rows.append({'transition_year': year, **fit_normal(subset)})
grw_y_params = pd.DataFrame(grw_y_rows)
grw_y_global_params = pd.DataFrame([{'model': 'global', **fit_normal(working_df_fit['log_delta'])}])
grw_y_initialization_params = shared_initialization_params.copy()
save_tables('grw_y', {'yearwise-parameters.csv': grw_y_params, 'global-parameters.csv': grw_y_global_params, 'initialization-parameters.csv': grw_y_initialization_params})
print('GRW-Y')
display(grw_y_params)


GRW-Y


,transition_year,n,mu,sigma,var
0,0,1963,0.279975,1.130057,1.277029
1,1,2003,0.180069,1.021314,1.043082
2,2,2005,0.071794,0.969837,0.940583
3,3,1944,0.044680,0.967429,0.935920
4,4,1874,-0.062821,0.963146,0.927651
5,5,1792,-0.047487,0.944065,0.891259
6,6,1717,-0.086533,0.968049,0.937119
7,7,1631,-0.017205,0.940605,0.884739
8,8,1550,-0.029842,0.915864,0.838807
9,9,1444,-0.042586,0.939586,0.882822


In [7]:
# Cell 7: full fit, AR(1)-GRW-Y

ar1_y_rows = []
for year in range(Y):
    subset = working_df_fit.loc[working_df_fit['CareerAge'].eq(year)]
    ar1_y_rows.append({'transition_year': year, **fit_log_ar1(subset)})
ar1_y_params = pd.DataFrame(ar1_y_rows)
ar1_y_global_params = pd.DataFrame([{'model': 'global', **fit_log_ar1(working_df_fit)}])
ar1_y_initialization_params = shared_initialization_params.copy()
save_tables('ar1_y', {'yearwise-parameters.csv': ar1_y_params, 'global-parameters.csv': ar1_y_global_params, 'initialization-parameters.csv': ar1_y_initialization_params})
print('AR(1)-GRW-Y')
display(ar1_y_params)


AR(1)-GRW-Y


,transition_year,n,intercept,beta,sigma_resid,var_resid,implied_stationary_mean_log,implied_stationary_var_log,mean_resid,sd_resid
0,0,1963,0.979817,0.402023,0.936911,0.877802,1.638552,1.047024,-2.624266e-17,0.936911
1,1,2003,0.928713,0.484905,0.875755,0.766947,1.802994,1.002720,4.394333e-16,0.875755
2,2,2005,0.835426,0.534576,0.850810,0.723878,1.794980,1.013511,-6.378937e-17,0.850810
3,3,1944,0.845369,0.535518,0.848332,0.719667,1.820026,1.009039,-1.135808e-15,0.848332
4,4,1874,0.745726,0.545489,0.847330,0.717968,1.640721,1.022104,1.251223e-16,0.847330
5,5,1792,0.721504,0.553210,0.828151,0.685835,1.614862,0.988294,8.931348e-16,0.828151
6,6,1717,0.617628,0.579002,0.871371,0.759287,1.467057,1.142203,1.738078e-16,0.871371
7,7,1631,0.655159,0.579168,0.830181,0.689201,1.556817,1.037071,-3.289146e-16,0.830181
8,8,1550,0.569835,0.619071,0.826164,0.682547,1.495911,1.106682,-6.200058e-16,0.826164
9,9,1444,0.545532,0.621992,0.852686,0.727073,1.443175,1.185845,-7.645469e-16,0.852686


In [8]:
# Cell 8: full fit, AR(1)-GRW-S

ar1_s_rows = []
for stage in CANONICAL_STAGE_ORDER:
    start, end = CANONICAL_STAGE_BOUNDS[stage]
    subset = working_df_fit.loc[working_df_fit['fit_stage'].eq(stage)]
    ar1_s_rows.append({'stage': stage, 'transition_year_start': start, 'transition_year_end': end, **fit_log_ar1(subset)})
ar1_s_params = pd.DataFrame(ar1_s_rows)
ar1_s_global_params = pd.DataFrame([{'model': 'global', **fit_log_ar1(working_df_fit)}])
ar1_s_initialization_params = shared_initialization_params.copy()
save_tables('ar1_s', {'stagewise-parameters.csv': ar1_s_params, 'global-parameters.csv': ar1_s_global_params, 'initialization-parameters.csv': ar1_s_initialization_params})
print('AR(1)-GRW-S')
display(ar1_s_params)


AR(1)-GRW-S


,stage,transition_year_start,transition_year_end,n,intercept,beta,sigma_resid,var_resid,implied_stationary_mean_log,implied_stationary_var_log,mean_resid,sd_resid
0,0,0,0,1963,0.979817,0.402023,0.936911,0.877802,1.638552,1.047024,-2.624266e-17,0.936911
1,1-4,1,4,7826,0.848278,0.521572,0.856860,0.734209,1.773055,1.008581,-1.089511e-16,0.856860
2,5-7,5,7,5140,0.664463,0.570536,0.843930,0.712219,1.547190,1.055938,1.347819e-16,0.843930
3,8-20,8,19,12104,0.482056,0.652411,0.847893,0.718923,1.386858,1.251694,-7.038506e-16,0.847893


In [9]:
# Cell 9: full fit, Hurdle-AR(1)-GRW-S

hurdle_ar1_s_positive = working_df_fit.loc[working_df_fit['state'].eq(1) & working_df_fit['next_state'].eq(1)].copy()
hurdle_ar1_s_transition_rows = []
hurdle_ar1_s_intensity_rows = []
for stage in CANONICAL_STAGE_ORDER:
    start, end = CANONICAL_STAGE_BOUNDS[stage]
    transition_subset = working_df_fit.loc[working_df_fit['fit_stage'].eq(stage)]
    positive_subset = hurdle_ar1_s_positive.loc[hurdle_ar1_s_positive['fit_stage'].eq(stage)]
    hurdle_ar1_s_transition_rows.append({'stage': stage, 'transition_year_start': start, 'transition_year_end': end, **transition_probabilities(transition_subset, working_df_fit)})
    fit = fit_positive_arp(positive_subset, order=1)
    fit['beta'] = fit.pop('beta_1')
    hurdle_ar1_s_intensity_rows.append({'stage': stage, 'transition_year_start': start, 'transition_year_end': end, **fit})
hurdle_ar1_s_hurdle_params = pd.DataFrame(hurdle_ar1_s_transition_rows)
hurdle_ar1_s_intensity_params = pd.DataFrame(hurdle_ar1_s_intensity_rows)
hurdle_ar1_s_global_fit = fit_positive_arp(hurdle_ar1_s_positive, order=1)
hurdle_ar1_s_global_fit['beta'] = hurdle_ar1_s_global_fit.pop('beta_1')
hurdle_ar1_s_global_params = pd.DataFrame([{'model': 'global', **hurdle_ar1_s_global_fit}])
hurdle_ar1_s_initialization_params = hurdle_initialization(working_df, working_df_fit)
save_tables('hurdle_ar1_s', {'stagewise-hurdle-parameters.csv': hurdle_ar1_s_hurdle_params, 'stagewise-positive-ar1-parameters.csv': hurdle_ar1_s_intensity_params, 'global-positive-ar1-parameters.csv': hurdle_ar1_s_global_params, 'initialization-parameters.csv': hurdle_ar1_s_initialization_params})
print('Hurdle-AR(1)-GRW-S')
display(hurdle_ar1_s_hurdle_params)
display(hurdle_ar1_s_intensity_params)


Hurdle-AR(1)-GRW-S


,stage,transition_year_start,transition_year_end,n_from_0,p_00,p_01,n_from_1,p_10,p_11
0,0,0,0,357,0.271709,0.728291,1606,0.096513,0.903487
1,1-4,1,4,779,0.338896,0.661104,7047,0.063999,0.936001
2,5-7,5,7,498,0.331325,0.668675,4642,0.075399,0.924601
3,8-20,8,19,1629,0.462247,0.537753,10475,0.088783,0.911217


,stage,transition_year_start,transition_year_end,n,intercept,sigma_resid,var_resid,spectral_radius,is_stationary,implied_stationary_mean_log,implied_stationary_var_log,beta
0,0,0,0,1451,1.020303,0.636288,0.404862,0.470656,True,1.927485,0.520065,0.470656
1,1-4,1,4,6596,0.875065,0.624009,0.389388,0.551970,True,1.953141,0.560005,0.551970
2,5-7,5,7,4292,0.718687,0.628191,0.394624,0.586939,True,1.739906,0.602017,0.586939
3,8-20,8,19,9545,0.619050,0.643390,0.413951,0.633217,True,1.687784,0.691028,0.633217


In [10]:
# Cell 10: full fit, Hurdle-AR(1)-GRW-S-P

hurdle_ar1_s_p_positive = working_df_fit.loc[working_df_fit['state'].eq(1) & working_df_fit['next_state'].eq(1)].copy()
hurdle_ar1_s_p_transition_rows = []
hurdle_ar1_s_p_intensity_rows = []
for stage in CANONICAL_STAGE_ORDER:
    start, end = CANONICAL_STAGE_BOUNDS[stage]
    transition_subset = working_df_fit.loc[working_df_fit['fit_stage'].eq(stage)]
    positive_subset = hurdle_ar1_s_p_positive.loc[hurdle_ar1_s_p_positive['fit_stage'].eq(stage)]
    hurdle_ar1_s_p_transition_rows.append({'stage': stage, 'transition_year_start': start, 'transition_year_end': end, **transition_probabilities(transition_subset, working_df_fit)})
    fit = fit_positive_arp(positive_subset, order=1)
    fit['beta'] = fit.pop('beta_1')
    hurdle_ar1_s_p_intensity_rows.append({'stage': stage, 'transition_year_start': start, 'transition_year_end': end, **fit})
hurdle_ar1_s_p_hurdle_params = pd.DataFrame(hurdle_ar1_s_p_transition_rows)
hurdle_ar1_s_p_intensity_params = pd.DataFrame(hurdle_ar1_s_p_intensity_rows)
hurdle_ar1_s_p_global_fit = fit_positive_arp(hurdle_ar1_s_p_positive, order=1)
hurdle_ar1_s_p_global_fit['beta'] = hurdle_ar1_s_p_global_fit.pop('beta_1')
hurdle_ar1_s_p_global_params = pd.DataFrame([{'model': 'global', **hurdle_ar1_s_p_global_fit}])
hurdle_ar1_s_p_dropout_params, hurdle_ar1_s_p_dropout_optimizer = fit_dropout_parameters(working_df_fit)
hurdle_ar1_s_p_initialization_params = hurdle_initialization(working_df, working_df_fit)
save_tables('hurdle_ar1_s_p', {'stagewise-hurdle-parameters.csv': hurdle_ar1_s_p_hurdle_params, 'stagewise-positive-ar1-parameters.csv': hurdle_ar1_s_p_intensity_params, 'global-positive-ar1-parameters.csv': hurdle_ar1_s_p_global_params, 'dropout-parameters.csv': hurdle_ar1_s_p_dropout_params, 'dropout-optimizer.csv': hurdle_ar1_s_p_dropout_optimizer, 'initialization-parameters.csv': hurdle_ar1_s_p_initialization_params})
print('Hurdle-AR(1)-GRW-S-P')
display(hurdle_ar1_s_p_hurdle_params)
display(hurdle_ar1_s_p_intensity_params)
display(hurdle_ar1_s_p_dropout_params)


Hurdle-AR(1)-GRW-S-P


,stage,transition_year_start,transition_year_end,n_from_0,p_00,p_01,n_from_1,p_10,p_11
0,0,0,0,357,0.271709,0.728291,1606,0.096513,0.903487
1,1-4,1,4,779,0.338896,0.661104,7047,0.063999,0.936001
2,5-7,5,7,498,0.331325,0.668675,4642,0.075399,0.924601
3,8-20,8,19,1629,0.462247,0.537753,10475,0.088783,0.911217


,stage,transition_year_start,transition_year_end,n,intercept,sigma_resid,var_resid,spectral_radius,is_stationary,implied_stationary_mean_log,implied_stationary_var_log,beta
0,0,0,0,1451,1.020303,0.636288,0.404862,0.470656,True,1.927485,0.520065,0.470656
1,1-4,1,4,6596,0.875065,0.624009,0.389388,0.551970,True,1.953141,0.560005,0.551970
2,5-7,5,7,4292,0.718687,0.628191,0.394624,0.586939,True,1.739906,0.602017,0.586939
3,8-20,8,19,9545,0.619050,0.643390,0.413951,0.633217,True,1.687784,0.691028,0.633217


,stage,dropout_intercept,dropout_gamma_positive,productivity_scale,p_dropout_at_scale
0,0,-0.979976,1.824037,6.00827,0.057108
1,1-4,-1.048910,1.824037,6.00827,0.053507
2,5-7,-0.852129,1.824037,6.00827,0.064394
3,8-20,-0.819778,1.824037,6.00827,0.066371


In [11]:
# Cell 11: full fit, Hurdle-AR(3)-GRW-S-P

positive_ar1 = working_df_fit.loc[working_df_fit['state'].eq(1) & working_df_fit['next_state'].eq(1)].copy()
positive_ar2 = working_df_fit_ar2.loc[working_df_fit_ar2['state_lag_1'].eq(1) & working_df_fit_ar2['state'].eq(1) & working_df_fit_ar2['next_state'].eq(1)].copy()
positive_ar3 = working_df_fit_ar3.loc[working_df_fit_ar3['state_lag_2'].eq(1) & working_df_fit_ar3['state_lag_1'].eq(1) & working_df_fit_ar3['state'].eq(1) & working_df_fit_ar3['next_state'].eq(1)].copy()
global_ar1 = fit_positive_arp(positive_ar1, order=1)
global_ar2 = fit_positive_arp(positive_ar2, order=2)
global_ar3 = fit_positive_arp(positive_ar3, order=3)
transition_rows = []
ar1_rows = []
ar2_rows = []
ar3_rows = []
for stage in CANONICAL_STAGE_ORDER:
    start, end = CANONICAL_STAGE_BOUNDS[stage]
    transition_subset = working_df_fit.loc[working_df_fit['fit_stage'].eq(stage)]
    ar1_subset = positive_ar1.loc[positive_ar1['fit_stage'].eq(stage)]
    ar2_subset = positive_ar2.loc[positive_ar2['fit_stage'].eq(stage)]
    ar3_subset = positive_ar3.loc[positive_ar3['fit_stage'].eq(stage)]
    transition_rows.append({'stage': stage, 'transition_year_start': start, 'transition_year_end': end, **transition_probabilities(transition_subset, working_df_fit)})
    ar1_rows.append({'stage': stage, 'transition_year_start': start, 'transition_year_end': end, **fit_positive_arp(ar1_subset, order=1)})
    ar2_rows.append({'stage': stage, 'transition_year_start': start, 'transition_year_end': end, **fit_positive_arp(ar2_subset, order=2)})
    ar3_rows.append({'stage': stage, 'transition_year_start': start, 'transition_year_end': end, **fit_positive_arp(ar3_subset, order=3)})
hurdle_ar3_s_p_hurdle_params = pd.DataFrame(transition_rows)
hurdle_ar3_s_p_ar1_params = pd.DataFrame(ar1_rows)
hurdle_ar3_s_p_ar2_params = pd.DataFrame(ar2_rows)
hurdle_ar3_s_p_ar3_params = pd.DataFrame(ar3_rows)
hurdle_ar3_s_p_global_ar1_params = pd.DataFrame([{'model': 'global', **global_ar1}])
hurdle_ar3_s_p_global_ar2_params = pd.DataFrame([{'model': 'global', **global_ar2}])
hurdle_ar3_s_p_global_ar3_params = pd.DataFrame([{'model': 'global', **global_ar3}])
hurdle_ar3_s_p_dropout_params, hurdle_ar3_s_p_dropout_optimizer = fit_dropout_parameters(working_df_fit)
hurdle_ar3_s_p_initialization_params = hurdle_initialization(working_df, working_df_fit)
save_tables('hurdle_ar3_s_p', {'stagewise-hurdle-parameters.csv': hurdle_ar3_s_p_hurdle_params, 'stagewise-positive-ar1-warm-start-parameters.csv': hurdle_ar3_s_p_ar1_params, 'stagewise-positive-ar2-warm-start-parameters.csv': hurdle_ar3_s_p_ar2_params, 'stagewise-positive-ar3-parameters.csv': hurdle_ar3_s_p_ar3_params, 'global-positive-ar1-warm-start-parameters.csv': hurdle_ar3_s_p_global_ar1_params, 'global-positive-ar2-warm-start-parameters.csv': hurdle_ar3_s_p_global_ar2_params, 'global-positive-ar3-parameters.csv': hurdle_ar3_s_p_global_ar3_params, 'dropout-parameters.csv': hurdle_ar3_s_p_dropout_params, 'dropout-optimizer.csv': hurdle_ar3_s_p_dropout_optimizer, 'initialization-parameters.csv': hurdle_ar3_s_p_initialization_params})
print('Hurdle-AR(3)-GRW-S-P')
display(hurdle_ar3_s_p_hurdle_params)
display(hurdle_ar3_s_p_ar1_params)
display(hurdle_ar3_s_p_ar2_params)
display(hurdle_ar3_s_p_ar3_params)
display(hurdle_ar3_s_p_dropout_params)


Hurdle-AR(3)-GRW-S-P


,stage,transition_year_start,transition_year_end,n_from_0,p_00,p_01,n_from_1,p_10,p_11
0,0,0,0,357,0.271709,0.728291,1606,0.096513,0.903487
1,1-4,1,4,779,0.338896,0.661104,7047,0.063999,0.936001
2,5-7,5,7,498,0.331325,0.668675,4642,0.075399,0.924601
3,8-20,8,19,1629,0.462247,0.537753,10475,0.088783,0.911217


,stage,transition_year_start,transition_year_end,n,intercept,beta_1,sigma_resid,var_resid,spectral_radius,is_stationary,implied_stationary_mean_log,implied_stationary_var_log
0,0,0,0,1451,1.020303,0.470656,0.636288,0.404862,0.470656,True,1.927485,0.520065
1,1-4,1,4,6596,0.875065,0.551970,0.624009,0.389388,0.551970,True,1.953141,0.560005
2,5-7,5,7,4292,0.718687,0.586939,0.628191,0.394624,0.586939,True,1.739906,0.602017
3,8-20,8,19,9545,0.619050,0.633217,0.643390,0.413951,0.633217,True,1.687784,0.691028


,stage,transition_year_start,transition_year_end,n,intercept,beta_1,beta_2,sigma_resid,var_resid,spectral_radius,is_stationary,implied_stationary_mean_log,implied_stationary_var_log
0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN
1,1-4,1,4,6004,0.698091,0.388305,0.286403,0.594298,0.353190,0.763448,True,2.146039,0.546597
2,5-7,5,7,4061,0.431808,0.404677,0.329904,0.591188,0.349504,0.811308,True,1.626887,0.617333
3,8-20,8,19,8939,0.364480,0.416151,0.357944,0.601121,0.361346,0.841510,True,1.613420,0.714691


,stage,transition_year_start,transition_year_end,n,intercept,beta_1,beta_2,beta_3,sigma_resid,var_resid,spectral_radius,is_stationary,implied_stationary_mean_log,implied_stationary_var_log
0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN
1,1-4,1,4,4181,0.587020,0.354709,0.274434,0.103485,0.574770,0.330360,0.833171,True,2.195516,0.544310
2,5-7,5,7,3880,0.318209,0.346091,0.270406,0.173308,0.580699,0.337212,0.878476,True,1.513870,0.631761
3,8-20,8,19,8445,0.269156,0.350075,0.291823,0.182308,0.587905,0.345633,0.899675,True,1.531090,0.732409


,stage,dropout_intercept,dropout_gamma_positive,productivity_scale,p_dropout_at_scale
0,0,-0.979976,1.824037,6.00827,0.057108
1,1-4,-1.048910,1.824037,6.00827,0.053507
2,5-7,-0.852129,1.824037,6.00827,0.064394
3,8-20,-0.819778,1.824037,6.00827,0.066371


In [12]:
# Cell 12: output registry and final audit

registry_rows = []
for model_dir in ['arw4', 'unfitted_grw', 'grw_y', 'ar1_y', 'ar1_s', 'hurdle_ar1_s', 'hurdle_ar1_s_p', 'hurdle_ar3_s_p']:
    for path in sorted((OUTPUT / model_dir).glob('*.csv')):
        frame = pd.read_csv(path)
        registry_rows.append({'model_dir': model_dir, 'file': path.name, 'rows': len(frame), 'columns': ','.join(frame.columns)})
fit_registry = pd.DataFrame(registry_rows)
fit_registry.to_csv(OUTPUT / 'fit-registry.csv', index=False)
expected_models = {'arw4', 'unfitted_grw', 'grw_y', 'ar1_y', 'ar1_s', 'hurdle_ar1_s', 'hurdle_ar1_s_p', 'hurdle_ar3_s_p'}
if set(fit_registry['model_dir']) != expected_models:
    raise AssertionError('One or more model output directories are missing.')
if grw_y_params['transition_year'].tolist() != list(range(Y)):
    raise AssertionError('GRW-Y does not cover transition years 0-19.')
if ar1_y_params['transition_year'].tolist() != list(range(Y)):
    raise AssertionError('AR(1)-GRW-Y does not cover transition years 0-19.')
for table in [ar1_s_params, hurdle_ar1_s_hurdle_params, hurdle_ar1_s_intensity_params, hurdle_ar1_s_p_hurdle_params, hurdle_ar1_s_p_intensity_params, hurdle_ar1_s_p_dropout_params, hurdle_ar3_s_p_hurdle_params, hurdle_ar3_s_p_ar1_params, hurdle_ar3_s_p_ar2_params, hurdle_ar3_s_p_ar3_params, hurdle_ar3_s_p_dropout_params]:
    if table['stage'].tolist() != CANONICAL_STAGE_ORDER:
        raise AssertionError('A stagewise table does not use the canonical stage order.')
print(f'Saved {len(fit_registry)} parameter files to {OUTPUT}')
display(fit_registry)


Saved 33 parameter files to /Users/samlunemagid/Desktop/SPAARW2/fit/output


,model_dir,file,rows,columns
0,arw4,global-parameters.csv,1,"stage,n,alpha,mode_beta,nll,mean_q_now,mean_q_..."
1,arw4,initialization-parameters.csv,1,"distribution,n,scale"
2,arw4,stage-parameters.csv,4,"stage,transition_year_start,transition_year_en..."
3,unfitted_grw,fixed-parameters.csv,4,"parameter,value"
4,grw_y,global-parameters.csv,1,"model,n,mu,sigma,var"
5,grw_y,initialization-parameters.csv,1,"distribution,n,scale"
6,grw_y,yearwise-parameters.csv,20,"transition_year,n,mu,sigma,var"
7,ar1_y,global-parameters.csv,1,"model,n,intercept,beta,sigma_resid,var_resid,i..."
8,ar1_y,initialization-parameters.csv,1,"distribution,n,scale"
9,ar1_y,yearwise-parameters.csv,20,"transition_year,n,intercept,beta,sigma_resid,v..."
